# Дообучение ruBERT для multi-label классификации новостей

Ноутбук рассчитан на файлы `train.jsonl`, `val.jsonl`, `test_relabeled.json` и `label_map.json` в `/content/drive/MyDrive/alfahack-dataset`. Сначала выполните установочную ячейку, перезапустите runtime Colab, затем запускайте остальные ячейки сверху вниз. По EDA используем 31 независимую метку, `max_length=512`, взвешенный BCE для редких классов и отдельные пороги на validation. Ручной тест не участвует в подборе параметров.

In [ ]:
# Выполните эту ячейку в новом Colab runtime, затем перезапустите runtime.
# torch и torchvision не переустанавливаем: Colab уже содержит CUDA-сборку.
%pip install -q --no-cache-dir --force-reinstall --only-binary=tokenizers 'transformers==4.57.1' 'huggingface-hub>=0.23,<1.0' 'tokenizers>=0.22,<0.24' 'safetensors>=0.4.1' 'pandas==2.2.3' scikit-learn matplotlib

In [ ]:
from google.colab import drive
import json, os, random
import pandas as pd
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support, average_precision_score

SEED = 42
MODEL_NAME = 'ai-forever/ruBert-base'
MAX_LENGTH = 512  # сохраняем начало длинного поста согласно документации
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 6
LR_ENCODER = 2e-5
LR_HEAD = 8e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/alfahack-dataset'
OUT_DIR = Path(DATA_DIR) / 'rubert_multilabel'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
print('device:', DEVICE, '| data:', DATA_DIR)

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f: return json.load(f)

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]

label_map = load_json(os.path.join(DATA_DIR, 'label_map.json'))
split_manifest = load_json(os.path.join(DATA_DIR, 'split_manifest.json'))
test_relabeled = load_json(os.path.join(DATA_DIR, 'test_relabeled.json'))
train = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))
train_df = pd.DataFrame(train); val_df = pd.DataFrame(val)
test_df = pd.DataFrame(test_relabeled if isinstance(test_relabeled, list) else test_relabeled.get('data', test_relabeled))
print('train:', train_df.shape, '| val:', val_df.shape, '| test:', test_df.shape)
print('labels:', len(label_map)); print('колонки train:', list(train_df.columns))
train_rows, val_rows = train, val
test_rows = test_relabeled if isinstance(test_relabeled, list) else test_relabeled.get('data', test_relabeled)
LABEL_NAMES = [label_map[str(i)] for i in range(len(label_map))]
LABEL2ID = {name: i for i, name in enumerate(LABEL_NAMES)}
N_LABELS = len(LABEL_NAMES)

def vector_from_row(row, test=False):
    if test:
        return [int(row['target'].get(name, 0)) for name in LABEL_NAMES]
    labels = row.get('labels', [])
    # В текущих train/val split labels уже является бинарным вектором.
    if len(labels) == N_LABELS and all(value in (0, 1) for value in labels):
        return [int(value) for value in labels]
    # Оставляем поддержку альтернативного формата: список названий категорий.
    return [int(name in set(labels)) for name in LABEL_NAMES]

def describe(rows, test=False):
    y = np.asarray([vector_from_row(r, test) for r in rows])
    return len(rows), float(y.sum(axis=1).sum()), y.sum(axis=0).astype(int)
for name, rows, is_test in [('train', train_rows, False), ('val', val_rows, False), ('test_relabeled', test_rows, True)]:
    n, positives, counts = describe(rows, is_test)
    print(name, n, 'avg labels:', round(positives / n, 3), 'min/max positives:', counts.min(), counts.max())

## Датасет и архитектура

Используем ruBERT encoder и голову, которая объединяет `CLS` с masked mean pooling. Mean pooling снижает зависимость от одного позиционного представления на длинных Telegram-постах; независимые выходы остаются multi-label logits.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(Dataset):
    def __init__(self, rows, test=False):
        self.rows, self.test = rows, test
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        return {'text': row['text'], 'labels': torch.tensor(vector_from_row(row, self.test), dtype=torch.float32), 'post_id': row['post_id']}

def collate(batch):
    texts = [x['text'] for x in batch]
    ids = [x['post_id'] for x in batch]
    labels = torch.stack([x['labels'] for x in batch])
    out = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
    out['labels'] = labels; out['post_id'] = ids
    return out

train_loader = DataLoader(NewsDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(NewsDataset(val_rows), batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
test_loader = DataLoader(NewsDataset(test_rows, test=True), batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)

class RuBertMultiLabel(nn.Module):
    def __init__(self, name, n_labels):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(nn.LayerNorm(2*h), nn.Dropout(0.2), nn.Linear(2*h, h), nn.GELU(), nn.Dropout(0.2), nn.Linear(h, n_labels))
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None: kw['token_type_ids'] = token_type_ids
        hidden = self.encoder(**kw).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return self.head(torch.cat([hidden[:, 0], mean], dim=-1))

model = RuBertMultiLabel(MODEL_NAME, N_LABELS).to(DEVICE)
pos = np.asarray([vector_from_row(r) for r in train_rows]).sum(axis=0)
neg = len(train_rows) - pos
pos_weight = torch.tensor(np.minimum(np.sqrt(neg / np.maximum(pos, 1)), 10.0), dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW([{'params': model.encoder.parameters(), 'lr': LR_ENCODER}, {'params': model.head.parameters(), 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
steps = max(1, (len(train_loader) * EPOCHS) // GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
print('labels:', N_LABELS, '| train steps:', steps, '| max_length:', MAX_LENGTH)

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval(); all_logits=[]; all_y=[]
    for b in loader:
        y = b.pop('labels').numpy(); b.pop('post_id')
        b = {k:v.to(DEVICE) for k,v in b.items()}
        all_logits.append(model(**b).float().cpu().numpy()); all_y.append(y)
    return np.concatenate(all_logits), np.concatenate(all_y)

def macro_f1(logits, y, thresholds=None):
    t = np.full(N_LABELS, 0.5) if thresholds is None else np.asarray(thresholds)
    return f1_score(y, (1/(1+np.exp(-logits)) >= t).astype(int), average='macro', zero_division=0)

best_f1, bad_epochs = -1, 0
history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
for epoch in range(EPOCHS):
    model.train(); optimizer.zero_grad(); running=0.0
    for step, b in enumerate(train_loader):
        y = b.pop('labels').to(DEVICE); b.pop('post_id'); b = {k:v.to(DEVICE) for k,v in b.items()}
        loss = criterion(model(**b), y) / GRAD_ACCUM; loss.backward(); running += loss.item()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); scheduler.step(); optimizer.zero_grad()
    val_logits, val_y = predict(val_loader); score = macro_f1(val_logits, val_y)
    val_loss = nn.functional.binary_cross_entropy_with_logits(torch.from_numpy(val_logits), torch.from_numpy(val_y), pos_weight=pos_weight.detach().cpu()).item()
    train_loss = running * GRAD_ACCUM / len(train_loader)
    history['epoch'].append(epoch + 1); history['train_loss'].append(float(train_loss)); history['val_loss'].append(float(val_loss)); history['val_macro_f1'].append(float(score))
    print(f'epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_macro_f1@0.5={score:.4f}')
    if score > best_f1:
        best_f1, bad_epochs = score, 0; torch.save(model.state_dict(), OUT_DIR / 'best_model.pt')
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE: print('early stopping'); break
model.load_state_dict(torch.load(OUT_DIR / 'best_model.pt', map_location=DEVICE))
(OUT_DIR / 'training_history.json').write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = history['epoch']
axes[0].plot(epochs, history['train_loss'], marker='o', label='train loss')
axes[0].plot(epochs, history['val_loss'], marker='o', label='validation loss')
axes[0].set_title('Loss по эпохам'); axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('BCE loss')
axes[0].grid(alpha=0.25); axes[0].legend()
axes[1].plot(epochs, history['val_macro_f1'], marker='o', color='tab:green', label='val macro-F1')
best_epoch = epochs[int(np.argmax(history['val_macro_f1']))]
axes[1].axvline(best_epoch, color='tab:red', linestyle='--', alpha=0.7, label=f'best epoch = {best_epoch}')
axes[1].set_title('Validation macro-F1'); axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('macro-F1'); axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.25); axes[1].legend(); fig.tight_layout()
fig.savefig(OUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight'); plt.show()
print('График сохранен:', OUT_DIR / 'training_curves.png')

## Пороги и финальная оценка

Порог 0.5 плохо отражает дисбаланс. Для каждой метки выбираем порог на `val` по F1; при менее чем 10 положительных примерах используем 0.5, чтобы не переобучить порог на малой выборке.

In [ ]:
val_logits, val_y = predict(val_loader); val_p = 1/(1+np.exp(-val_logits))
thresholds = np.full(N_LABELS, 0.5)
for j in range(N_LABELS):
    if val_y[:, j].sum() < 10: continue
    candidates = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(val_y[:,j], val_p[:,j] >= t, zero_division=0) for t in candidates]
    thresholds[j] = candidates[int(np.argmax(scores))]
thresholds_payload = {LABEL_NAMES[i]: float(thresholds[i]) for i in range(N_LABELS)}
(OUT_DIR / 'thresholds.json').write_text(json.dumps(thresholds_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('val macro-F1:', macro_f1(val_logits, val_y, thresholds))

test_logits, test_y = predict(test_loader); test_p = 1/(1+np.exp(-test_logits))
test_pred = (test_p >= thresholds).astype(int)
macro = f1_score(test_y, test_pred, average='macro', zero_division=0)
micro = f1_score(test_y, test_pred, average='micro', zero_division=0)
pr, rc, f1, support = precision_recall_fscore_support(test_y, test_pred, zero_division=0)
ap = [float(average_precision_score(test_y[:,j], test_p[:,j])) if test_y[:,j].sum() else None for j in range(N_LABELS)]
print(f"test_relabeled: macro-F1={macro:.4f}, micro-F1={micro:.4f}, sample-F1={f1_score(test_y, test_pred, average='samples', zero_division=0):.4f}")
for i, name in enumerate(LABEL_NAMES): print(f"{name}: P={pr[i]:.3f} R={rc[i]:.3f} F1={f1[i]:.3f} AP={ap[i] if ap[i] is not None else 'n/a'} n={int(test_y[:,i].sum())}")
metrics = {'macro_f1': float(macro), 'micro_f1': float(micro), 'thresholds': thresholds_payload, 'per_label': {name: {'precision': float(pr[i]), 'recall': float(rc[i]), 'f1': float(f1[i]), 'average_precision': float(ap[i])} for i,name in enumerate(LABEL_NAMES)}}
(OUT_DIR / 'test_metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
# Сохраняем токенизатор, конфигурацию и карту меток для инференса.
tokenizer.save_pretrained(OUT_DIR / 'tokenizer')
(OUT_DIR / 'label_map.json').write_text(json.dumps(label_map, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / 'config.json').write_text(json.dumps({'base_model': MODEL_NAME, 'max_length': MAX_LENGTH, 'head': 'concat_cls_masked_mean_mlp', 'n_labels': N_LABELS, 'seed': SEED}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Artifacts saved to', OUT_DIR.resolve())